# 2. Collective movement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/02_collective_movement.ipynb)

Cells can bias their reorientation using neighboring occupancy.
This lesson compares random walk, polar alignment, aggregation and
nematic alignment using the same lattice, density and run length.

**Learning objectives**

- choose different standard models by name with `get_lgca`;
- separate an interaction mechanism from its parameter strength;
- compare mechanisms with a quantitative observable; and
- distinguish polar and nematic organization.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca import get_lgca

## One lattice, four mechanisms

The function below keeps the experimental controls in one place:
lattice, density, run length and seed are the same for every model,
and only the interaction and its strength `beta` change. This is how
models that differ by one named mechanism are compared.

In [ ]:
def run_collective_model(interaction, beta=2.0, seed=21, steps=25):
    parameters = {} if interaction == "random_walk" else {"beta": beta}
    lgca = get_lgca(
        geometry="hex",
        dims=(20, 20),
        bc="periodic",
        density=0.2,
        restchannels=0,
        interaction=interaction,
        seed=seed,
        **parameters,
    )
    lgca.timeevo(timesteps=steps, record=True, showprogress=False)
    return lgca


interaction_names = (
    "random_walk",
    "alignment",
    "aggregation",
    "nematic",
)

The four mechanisms answer different biological questions:

- **random walk:** what happens without a directional cue?
- **alignment:** can neighbor-induced orientation create streams?
- **aggregation:** does motion up a local density gradient form clusters?
- **nematic alignment:** can cells share an axis while moving in
  opposite directions along it?

In [ ]:
models = {name: run_collective_model(name, beta=2.0, seed=21) for name in interaction_names}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8), constrained_layout=True)
for axis, (name, lgca) in zip(axes.flat, models.items()):
    lgca.plot_density(ax=axis, vmax=6)
    axis.set_title(name.replace("_", " "))
plt.show()
plt.close(fig)

The density maps show where cells are, not where they go. An
animation of the flux shows the movement: the colour of a node is
the direction of its net flux. Watch how alignment turns random
headings into coherent streams.


In [ ]:
models["alignment"].animate_flux(figsize=(5, 4.5))

## A global polarization observable

Visual differences are useful for exploration but insufficient for
comparisons. Global polarization measures the magnitude of the
summed velocity vector divided by particle number. It approaches
one when most particles move in one direction and stays small for
disordered motion. A nematic state can be strongly ordered yet have
low polarization because opposite directions cancel; it therefore
needs a nematic observable in a detailed study.


In [ ]:
def polarization(lgca, nodes):
    total_particles = nodes.sum()
    if total_particles == 0:
        return 0.0
    total_flux = lgca.calc_flux(nodes).sum(axis=(0, 1))
    return float(np.linalg.norm(total_flux) / total_particles)


final_polarization = {
    name: polarization(lgca, lgca.data["nodes"][-1])
    for name, lgca in models.items()
}
final_polarization

In [ ]:
labels = [name.replace("_", " ") for name in interaction_names]
values = [final_polarization[name] for name in interaction_names]

fig, axis = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
axis.bar(labels, values)
axis.set_ylabel("final global polarization")
axis.tick_params(axis="x", rotation=25)
plt.show()
plt.close(fig)


## Interpretation and limitations

The controlled setup attributes differences to the interaction
rule, but a single seed and parameter value do not establish a
robust phase diagram. Aggregation is better quantified with a
clustering measure, and nematic alignment with an axis-sensitive
order parameter. Choosing an observable is part of choosing the
scientific question.

## Exercises

1. Run alignment for several values of `beta` and plot polarization
   against beta.
2. Define a density-variance or occupied-cluster observable for
   aggregation. Does it agree with the density maps?
3. Construct a nematic order parameter that treats directions
   separated by 180 degrees as equivalent.
4. Repeat each mechanism for five seeds and add uncertainty bars.